In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# ===== 合併 train/test（避免欄位不一致）=====
all_data = pd.concat([train.drop("SalePrice", axis=1), test], axis=0)

# ===== One-Hot Encoding =====
all_data = pd.get_dummies(all_data)

# ===== 再切回 train/test =====
X_train = all_data[:len(train)]
X_test = all_data[len(train):]

print(X_train.select_dtypes(include=["object"]).columns)

# 保持和原本一樣
train = train.drop(
    train[
        (train["GrLivArea"] > 4000) &
        (train["SalePrice"] < 300000)
    ].index
)

Index([], dtype='str')


In [2]:
# 資料前處理與特徵工程

import pandas as pd
import numpy as np

# 1. 目標變數轉換 (對數轉換以處理偏態)
# 提醒：最後輸出 Kaggle 預測檔案或報告計算真實房價誤差時，記得使用 np.expm1() 還原
y_train = np.log1p(train["SalePrice"])

# 為了方便一起處理特徵，先將 train 的 SalePrice 移除，並將 train 與 test 合併
# 同時先捨棄在預測中不需要的 Id 欄位
train_features = train.drop(["SalePrice", "Id"], axis=1)
test_features = test.drop(["Id"], axis=1)

# 將合併後的資料儲存至 all_data
all_data = pd.concat([train_features, test_features]).reset_index(drop=True)
print(f"合併後的資料維度: {all_data.shape}")

# 2. 缺失值填補 (Missing Value Imputation)
# 數值型特徵：LotFrontage 依 Neighborhood 中位數填補，其餘填 0
# 類別型特徵：全部填補為 'None' (代表無此設施)

# 處理 LotFrontage
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

# 區分數值型與類別型特徵
numeric_cols = all_data.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = all_data.select_dtypes(include=['object']).columns

# 其餘數值特徵填補為 0
numeric_cols = numeric_cols.drop("LotFrontage", errors="ignore")
all_data[numeric_cols] = all_data[numeric_cols].fillna(0)

# 類別特徵填補為 'None'
all_data[categorical_cols] = all_data[categorical_cols].fillna("None")

# 3. 特徵編碼 (One-Hot Encoding)
# 將所有文字類別轉換為數值表示
all_data = pd.get_dummies(all_data)
print(f"One-Hot Encoding 後的資料維度: {all_data.shape}")

# 4. 將資料切分回 train 與 test
# 透過原本 y_train 的長度來切分
X_train = all_data.iloc[:len(y_train), :]
X_test = all_data.iloc[len(y_train):, :]

print(f"處理完畢！X_train 維度: {X_train.shape}, X_test 維度: {X_test.shape}")

合併後的資料維度: (2917, 79)
One-Hot Encoding 後的資料維度: (2917, 309)
處理完畢！X_train 維度: (1458, 309), X_test 維度: (1459, 309)


C:\Users\austi\AppData\Local\Temp\ipykernel_9256\1947762553.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = all_data.select_dtypes(include=['object']).columns


In [3]:
X_train.to_csv(
    "../data/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/X_test.csv",
    index=False
)

pd.DataFrame(y_train).to_csv(
    "../data/y_train.csv",
    index=False
)

print("資料已儲存")

資料已儲存
